In [1]:
import datetime
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_scheduler,
    DataCollatorWithPadding
)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from tqdm.auto import tqdm
from sklearn.model_selection import StratifiedKFold

start_time = datetime.datetime.now()
timing_info = {}
print(f"Script started at {start_time}")

# Configuration metrics
file_name = "GroundTruth.csv"
checkpoint = "bert-base-cased"
num_folds = 5
emotions = ['Joy', 'Trust', 'Fear', 'Surprise', 'Sadness', 'Disgust', 'Anger', 'Anticipation', 'Neutral']
label_columns = ["Joy", "Trust", "Fear", "Surprise", "Sadness", "Disgust", "Anger", "Anticipation", "Neutral", "Reject"]

undersample = True
cut_off = 150
suffixes = []
if undersample:
    suffixes.append("undersample")

suffix_str = "_" + "_".join(suffixes) if suffixes else ""
output_path_csv = f"resultsBDS/binary_results_{checkpoint}_with_folds_{num_folds}__{suffix_str}.csv"

df_general = pd.read_csv(file_name, sep=";")
df_general = df_general[df_general['Reject'] != 1]


if undersample:
    start_undersampling = datetime.datetime.now()
    rng = np.random.default_rng(42)
    df_general = df_general.reset_index(drop=True)
    pos_counts = df_general[emotions].sum(axis=0).astype(int)
    targets = np.minimum(cut_off, pos_counts).astype(int)

    selected_idx = set()
    for lbl, tgt in zip(emotions, targets):
        if tgt <= 0:
            continue
        pos_idx = df_general.index[df_general[lbl] == 1].to_numpy()
        if len(pos_idx) <= tgt:
            chosen = pos_idx
        else:
            chosen = rng.choice(pos_idx, size=int(tgt), replace=False)
        selected_idx.update(int(x) for x in chosen)

    undersampled = df_general.loc[sorted(selected_idx)].reset_index(drop=True)
    print("Target per label:", dict(zip(emotions, targets.tolist())))
    print("Result per-label counts:\n", undersampled[emotions].sum().to_dict())
    print("Total rows selected:", len(undersampled))
    df_general = undersampled
    end_undersampling = datetime.datetime.now()
    timing_info['Undersampling time'] = end_undersampling - start_undersampling

non_label_columns = [col for col in df_general.columns if col not in emotions]
all_results = []
timing_info = {}


# Dataset for single-label
class EmotionDataset(Dataset):
    def __init__(self, df, tokenizer, label_col):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.label_col = label_col

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        tokenized = self.tokenizer(row["sentence"], truncation=True, padding=False)
        tokenized["labels"] = int(row[self.label_col])
        return tokenized


def collate_fn(batch):
    input_features = [{k: v for k, v in ex.items() if k != "labels"} for ex in batch]
    labels = [ex["labels"] for ex in batch]
    batch_inputs = data_collator(input_features)
    batch_inputs["labels"] = torch.tensor(labels, dtype=torch.long)
    return batch_inputs


for label in emotions:
    start_time_label = datetime.datetime.now()
    label_col = 'Emotion'
    df = df_general[non_label_columns + [label]].copy()
    df = df.rename(columns={label: label_col})
    output_filename = f"../binaryDS/{label}.csv"
    df.to_csv(output_filename, sep=";", index=False)

    print(f"Archivo generado: {output_filename}")
    timing_info[f'Generate {label} file'] = datetime.datetime.now() - start_time_label

    print("----------------------------------------------------")
    print(f"Processing label: {label}")
    print("----------------------------------------------------")

    num_labels = df[label_col].nunique()

    # Tokenizer and collator
    tokenizer = AutoTokenizer.from_pretrained(checkpoint)
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Cross-validation
    X = df["sentence"].values
    y = df[label_col].values
    skf = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=42)
    all_metrics = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        start_time_fold = datetime.datetime.now()
        print(f"\n===== Fold {fold + 1} =====")
        train_df = df.iloc[train_idx].reset_index(drop=True)
        val_df = df.iloc[val_idx].reset_index(drop=True)

        train_dataset = EmotionDataset(train_df, tokenizer, label_col)
        val_dataset = EmotionDataset(val_df, tokenizer, label_col)

        train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
        val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn)

        model = AutoModelForSequenceClassification.from_pretrained(
            checkpoint,
            num_labels=num_labels
        )
        model.to(device)

        optimizer = AdamW(model.parameters(), lr=2e-5)
        num_epochs = 3
        num_training_steps = num_epochs * len(train_loader)
        lr_scheduler = get_scheduler("linear", optimizer=optimizer,
                                     num_warmup_steps=0,
                                     num_training_steps=num_training_steps)

        model.train()
        progress_bar = tqdm(range(num_training_steps))
        for epoch in range(num_epochs):
            for batch in train_loader:
                batch = {k: v.to(device) for k, v in batch.items()}
                outputs = model(**batch)
                loss = outputs.loss
                loss.backward()
                optimizer.step()
                lr_scheduler.step()
                optimizer.zero_grad()
                progress_bar.update(1)

        # Evaluation
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for batch in val_loader:
                labels = batch["labels"].cpu().numpy()
                inputs = {k: v.to(device) for k, v in batch.items() if k != "labels"}
                outputs = model(**inputs)
                preds = outputs.logits.argmax(dim=-1).cpu().numpy()
                all_preds.append(preds)
                all_labels.append(labels)

        y_pred = np.concatenate(all_preds)
        y_true = np.concatenate(all_labels)

        global_metrics = {
            "accuracy": accuracy_score(y_true, y_pred),
            "precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
            "micro-precision": precision_score(y_true, y_pred, average="micro", zero_division=0),
            "recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
            "micro-recall": recall_score(y_true, y_pred, average="micro", zero_division=0),
            "f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
            "micro-f1": f1_score(y_true, y_pred, average="micro", zero_division=0)
        }
        print(f"Accuracy  -> {global_metrics['accuracy']}")
        print(f"Precision -> Macro: {global_metrics['precision']}, Micro: {global_metrics['micro-precision']}")
        print(f"Recall    -> Macro: {global_metrics['recall']}, Micro: {global_metrics['micro-recall']}")
        print(f"F1 Score  -> Macro: {global_metrics['f1']}, Micro: {global_metrics['micro-f1']}")

        result_row = {"emotion": label, "fold": fold + 1}
        result_row.update(global_metrics)
        all_results.append(result_row)

        report = classification_report(y_true, y_pred, zero_division=0, output_dict=True)
        df_report = pd.DataFrame(report).transpose()
        df_report["fold"] = fold + 1
        all_metrics.append((global_metrics, df_report))
        timing_info[f'{label} - Fold {fold + 1}'] = datetime.datetime.now() - start_time_fold

    # Average results
    global_results = [m[0] for m in all_metrics]
    df_global = pd.DataFrame(global_results)
    print(f"\n===== Global metrics averaged across folds {label}=====")
    print(df_global.mean())

# === Simplified results summary ===
df_summary = pd.DataFrame(all_results)

print("\n===== Results by Emotion =====")
emotion_tables = []

for emotion in df_summary["emotion"].unique():
    sub = df_summary[df_summary["emotion"] == emotion]
    # Build a simple table: metrics as rows, folds as columns
    metrics = ["accuracy", "precision", "recall", "f1"]
    table = pd.DataFrame(index=metrics)
    for fold in sorted(sub["fold"].unique()):
        fold_values = sub[sub["fold"] == fold].iloc[0]
        table[f"Fold {fold}"] = [fold_values[m] for m in metrics]

    print(f"\nEmotion: {emotion}")
    print(table)
    emotion_tables.append((emotion, table))

# === Timings summary ===
print("\n===== Timings =====")
timing_rows = []
for key, delta in timing_info.items():
    if "Fold" in key:
        emotion = key.split(" - ")[0]
        fold = key.split("Fold ")[1]
        timing_rows.append({
            "Emotion": emotion,
            "Fold": f"Fold {fold}",
            "Time (s)": delta.total_seconds()
        })

df_timing = pd.DataFrame(timing_rows)

for emotion in df_timing["Emotion"].unique():
    sub = df_timing[df_timing["Emotion"] == emotion].pivot(index="Emotion", columns="Fold", values="Time (s)")
    print(f"\nEmotion: {emotion}")
    print(sub)

# === Save all to CSV ===
with open(output_path_csv, "w", encoding="utf-8") as f:
    for emotion, table in emotion_tables:
        f.write(f"\nEmotion: {emotion}\n")
        table.to_csv(f, float_format="%.4f")
    f.write("\n\n===== Timings =====\n")
    df_timing.to_csv(f, float_format="%.2f", index=False)

print(f"\n✅ Results saved to {output_path_csv}")

/Users/carlotacatot/Docs/pHd/EmotionClassification/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Script started at 2025-11-26 11:07:37.654838
Target per label: {'Joy': 150, 'Trust': 150, 'Fear': 24, 'Surprise': 54, 'Sadness': 150, 'Disgust': 50, 'Anger': 27, 'Anticipation': 150, 'Neutral': 88}
Result per-label counts:
 {'Joy': 191, 'Trust': 150, 'Fear': 24, 'Surprise': 54, 'Sadness': 179, 'Disgust': 50, 'Anger': 27, 'Anticipation': 158, 'Neutral': 88}
Total rows selected: 776


OSError: Cannot save file into a non-existent directory: '../binaryDS'